# 🧭 Lab 5.4 — Agent-Specific Judges: Tool Call Efficiency & Knowledge Retention

## Learning Objectives
In this notebook, you will learn:
1. **Tool-Using Agent** — Build a small agent with `get_weather` + `get_time` tools, traced as TOOL spans
2. **`ToolCallEfficiency`** — Built-in agent judge that flags redundant or wasteful tool invocations
3. **Custom Tool-Count Scorer** — Trace-based `@scorer` that counts TOOL spans and enforces a hard SLA
4. **`KnowledgeRetention`** — Multi-turn judge that checks whether the agent remembers facts across turns
5. **v1 vs v2 Comparison** — Quantify behaviour change after a system-prompt rewrite — efficiency up, correctness held
6. **Module 5 Outcome** — Confirm: full RAG eval (5.2) + agent behaviour eval (5.4) covers all three quality dimensions

## Prerequisites
- Lab 1.3 (workspace + Foundation Model API)
- Lab 4.2 + 4.3 (judges + `@scorer` patterns)
- Conceptual familiarity with span types from Lab 5.3


In [ ]:
# ============================================================================
# 📦 INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-openai

dbutils.library.restartPython()


---
## Step 1 — Configure Namespace, Experiment, and Autolog


In [ ]:
# ============================================================================
# 🗂️ STEP 1 - CONFIGURE NAMESPACE, EXPERIMENT, AND AUTOLOG
# ============================================================================

import json
import mlflow
from mlflow.entities import SpanType

CATALOG = "genai_eval_tutorial"
SCHEMA  = "module_05"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.openai.autolog()

print(f"Experiment: {EXPERIMENT_PATH}")


---
## Step 2 — Build a Tool-Using Agent (v1)

Two tools, both deterministic so we can reason about *correct* behaviour:
- `get_weather(city)` → temperature in C
- `get_time(city)` → local time

We trace the dispatcher loop with `@mlflow.trace`. Each tool invocation gets its own span via `span_type=SpanType.TOOL` so trace-based scorers can count them.


In [ ]:
# ============================================================================
# ▶️ STEP 2 - BUILD A TOOL-USING AGENT (V1)
# ============================================================================

from databricks.sdk import WorkspaceClient

llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

WEATHER = {"berlin": 12, "tokyo": 18, "san francisco": 16, "sydney": 22}
LOCAL_TIME = {"berlin": "14:05", "tokyo": "22:05", "san francisco": "05:05", "sydney": "23:05"}

@mlflow.trace(span_type=SpanType.TOOL)
def get_weather(city: str) -> dict:
    return {"city": city, "temp_c": WEATHER.get(city.lower(), 20)}

@mlflow.trace(span_type=SpanType.TOOL)
def get_time(city: str) -> dict:
    return {"city": city, "local_time": LOCAL_TIME.get(city.lower(), "12:00")}

TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "get_weather", "description": "Current temperature in C for a city.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "get_time", "description": "Current local time for a city.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
]

TOOL_REGISTRY = {"get_weather": get_weather, "get_time": get_time}

# v1 system prompt: vague — invites redundant calls
SYSTEM_PROMPT_V1 = (
    "You are a helpful assistant. Use tools when needed to answer the user."
)

@mlflow.trace
def agent_v1(question: str) -> str:
    return _run_agent(question, SYSTEM_PROMPT_V1, max_iters=8)

def _run_agent(question: str, system_prompt: str, max_iters: int = 8) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ]
    for _ in range(max_iters):
        resp = llm_client.chat.completions.create(
            model="databricks-claude-sonnet-4",
            messages=messages,
            tools=TOOLS_SPEC,
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content or ""
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_REGISTRY[tc.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "(max tool iterations reached)"

print(agent_v1("What is the temperature in Berlin?"))


---
## Step 3 — Build a Tiny Tool-Use Eval Dataset

Each row pins the question, the **expected answer**, and the **expected number of tool calls**. The tool-call count gives our custom scorer something concrete to compare against.


In [ ]:
# ============================================================================
# 📚 STEP 3 - BUILD A TINY TOOL-USE EVAL DATASET
# ============================================================================

import pandas as pd

tool_eval_rows = [
    {"inputs": {"question": "What is the temperature in Berlin?"},
     "expectations": {"expected_response": "12", "expected_tool_calls": 1}},
    {"inputs": {"question": "What time is it in Tokyo right now?"},
     "expectations": {"expected_response": "22:05", "expected_tool_calls": 1}},
    {"inputs": {"question": "Tell me the temperature in San Francisco."},
     "expectations": {"expected_response": "16", "expected_tool_calls": 1}},
    {"inputs": {"question": "What's the weather in Sydney?"},
     "expectations": {"expected_response": "22", "expected_tool_calls": 1}},
    {"inputs": {"question": "Give me the temperature in Berlin and the local time in Tokyo."},
     "expectations": {"expected_response": "Berlin 12, Tokyo 22:05", "expected_tool_calls": 2}},
]
tool_eval_df = pd.DataFrame(tool_eval_rows)
display(tool_eval_df)


---
## Step 4 — Run the Built-in `ToolCallEfficiency` Judge

`ToolCallEfficiency` reads the trace, looks at each tool span, and judges whether the **set of calls was minimal and non-redundant** for the given question. No extra labelling needed — it's a behaviour judge over the trace.


In [ ]:
# ============================================================================
# ▶️ STEP 4 - RUN THE BUILT-IN `TOOLCALLEFFICIENCY` JUDGE
# ============================================================================

from mlflow.genai.scorers import ToolCallEfficiency, Correctness

results_v1 = mlflow.genai.evaluate(
    data=tool_eval_df,
    predict_fn=agent_v1,
    scorers=[ToolCallEfficiency(), Correctness()],
    model_id="models:/tool-agent/v1",
)

display(results_v1.tables["eval_results"])


---
## Step 5 — Custom Trace-Based Scorer: TOOL-Span Counter

`ToolCallEfficiency` gives a yes/no quality verdict. We complement it with a **deterministic counter** — useful for hard SLAs ("never make more than 3 tool calls for a single-fact question").


In [ ]:
# ============================================================================
# 🔭 STEP 5 - CUSTOM TRACE-BASED SCORER: TOOL-SPAN COUNTER
# ============================================================================

from mlflow.entities import Feedback, AssessmentSource
from mlflow.genai.scorers import scorer

MAX_TOOL_CALLS_FOR_SIMPLE_Q = 3

@scorer
def tool_call_count(trace) -> Feedback:
    spans = trace.search_spans(span_type=SpanType.TOOL)
    n = len(spans)
    return Feedback(
        value="ok" if n <= MAX_TOOL_CALLS_FOR_SIMPLE_Q else "too_many",
        rationale=f"{n} tool calls (limit: {MAX_TOOL_CALLS_FOR_SIMPLE_Q})",
        source=AssessmentSource(source_type="CODE", source_id="tool_count_v1"),
    )

print("Custom tool-count scorer ready.")


---
## Step 6 — Multi-Turn Eval with `KnowledgeRetention`

`KnowledgeRetention` checks whether the agent **remembers facts established earlier in a conversation**. It needs a multi-turn trace, so we build one explicitly: the user states a fact in turn 1 ("My name is Alex") and asks about it later.


In [ ]:
# ============================================================================
# ▶️ STEP 6 - MULTI-TURN EVAL WITH `KNOWLEDGERETENTION`
# ============================================================================

@mlflow.trace
def multi_turn_agent(messages: list[dict]) -> str:
    """Pass-through agent that takes a full message history and returns the next assistant turn."""
    full = [{"role": "system", "content": SYSTEM_PROMPT_V1}] + messages
    resp = llm_client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=full,
        tools=TOOLS_SPEC,
    )
    return resp.choices[0].message.content or ""

multi_turn_rows = [
    {"inputs": {"messages": [
        {"role": "user", "content": "Hi! My name is Alex and I'm planning a trip to Tokyo."},
        {"role": "assistant", "content": "Nice to meet you, Alex! Tokyo's a great choice."},
        {"role": "user", "content": "What time is it there right now?"},
        {"role": "assistant", "content": "It's 22:05 in Tokyo."},
        {"role": "user", "content": "Cool — and remind me, what city did I say I was visiting?"},
    ]}},
    {"inputs": {"messages": [
        {"role": "user", "content": "I prefer temperatures in Fahrenheit."},
        {"role": "assistant", "content": "Got it — I'll use Fahrenheit."},
        {"role": "user", "content": "What's the temperature in Berlin?"},
    ]}},
]
multi_turn_df = pd.DataFrame(multi_turn_rows)


In [ ]:
# ============================================================================
# ▶️ STEP
# ============================================================================

from mlflow.genai.scorers import KnowledgeRetention

results_multi = mlflow.genai.evaluate(
    data=multi_turn_df,
    predict_fn=multi_turn_agent,
    scorers=[KnowledgeRetention()],
    model_id="models:/tool-agent/v1-multiturn",
)

display(results_multi.tables["eval_results"])


---
## Step 7 — Optimize the Prompt: Agent v2

v1's vague prompt led the model to over-call tools (e.g., fetching the same city's weather twice "to be sure"). v2 adds explicit instructions:
- Call each tool **at most once per entity per turn**.
- Don't call a tool if the user's question doesn't need it.


In [ ]:
# ============================================================================
# ▶️ STEP 7 - OPTIMIZE THE PROMPT: AGENT V2
# ============================================================================

SYSTEM_PROMPT_V2 = (
    "You are a helpful assistant with access to weather and time tools. "
    "Rules:\n"
    "1. Call each tool at most ONCE per city per user turn.\n"
    "2. Do not call a tool if the user is making conversation, not asking for data.\n"
    "3. After tools return, answer concisely without re-checking."
)

@mlflow.trace
def agent_v2(question: str) -> str:
    return _run_agent(question, SYSTEM_PROMPT_V2, max_iters=8)

results_v2 = mlflow.genai.evaluate(
    data=tool_eval_df,
    predict_fn=agent_v2,
    scorers=[ToolCallEfficiency(), Correctness(), tool_call_count],
    model_id="models:/tool-agent/v2",
)

display(results_v2.tables["eval_results"])


In [ ]:
# ============================================================================
# ▶️ SIDE-BY-SIDE: V1 VS V2 ON TOOL-USE BEHAVIOUR
# ============================================================================

def behaviour_summary(results, label):
    return results.tables["eval_results"].selectExpr(
        f"'{label}' AS run",
        "AVG(CASE WHEN `tool_call_efficiency/v1/value` = 'yes' THEN 1.0 ELSE 0.0 END) AS efficiency_pass",
        "AVG(CASE WHEN `correctness/v1/value`         = 'yes' THEN 1.0 ELSE 0.0 END) AS correctness_pass",
    )

# v1 has no tool_call_count column; rerun v1 with the counter for a fair comparison.
results_v1_with_count = mlflow.genai.evaluate(
    data=tool_eval_df,
    predict_fn=agent_v1,
    scorers=[ToolCallEfficiency(), Correctness(), tool_call_count],
    model_id="models:/tool-agent/v1-with-count",
)

display(behaviour_summary(results_v1_with_count, "v1").union(behaviour_summary(results_v2, "v2")))


In [ ]:
# ============================================================================
# ▶️ PER-ROW TOOL-CALL COUNTS: V1 VS V2
# ============================================================================

def per_row_counts(results, label):
    return results.tables["eval_results"].selectExpr(
        f"'{label}' AS run",
        "inputs",
        "`tool_call_count/v1/value` AS count_bucket",
        "`tool_call_count/v1/rationale` AS detail",
    )

display(per_row_counts(results_v1_with_count, "v1").union(per_row_counts(results_v2, "v2")))


---
## Step 8 — Compare Runs in the MLflow UI

Open the experiment, select the **v1** and **v2** runs, click **Compare**. You'll see:
- `tool_call_efficiency` pass rate ↑
- `tool_call_count` distribution shift toward `ok`
- `correctness` ideally unchanged (we're optimising path, not destination)

If correctness drops with v2, the new prompt is too restrictive — iterate.


---
## Lab Complete

| Check | Status |
| --- | --- |
| Tool-using agent built; tool spans visible in trace | ✅ |
| `ToolCallEfficiency` applied to detect redundant calls | ✅ |
| Trace-based `tool_call_count` scorer flags > 3 calls | ✅ |
| `KnowledgeRetention` applied to multi-turn conversation | ✅ |
| v1 vs v2 system-prompt comparison quantified | ✅ |

**Module 5 Outcome:** You can evaluate full RAG pipelines across all three quality dimensions and diagnose failure modes from traces. You can evaluate agent tool-use behaviour and compare agent versions quantitatively.

Next: **Module 6** — Human review, labelling sessions, and judge calibration against human ground truth.


---
## 📝 Summary

In this notebook, we covered:

### 1. Why Agent-Specific Judges
- Final-answer judges miss path quality — a correct answer that took 8 tool calls is still expensive and fragile.
- Agent judges read the trace and grade *how* the agent reached the answer, not just whether it did.

### 2. Decision Matrix — Agent Eval
- **Redundant tool calls?** → `ToolCallEfficiency` (built-in agent judge).
- **Hard SLA on tool count?** → custom trace-based `@scorer` counting TOOL spans.
- **Multi-turn fact retention?** → `KnowledgeRetention` (built-in multi-turn judge).
- **Path-vs-destination question?** → compare versions in MLflow Experiments comparison view.

### 3. Module 5 Outcome — Coverage Map
- **RAG quality (5.2)** — retrieval recall + groundedness + correctness, with failure-mode diagnosis.
- **Agent behaviour (5.4)** — tool efficiency + multi-turn retention + version-over-version regression catch.
- Together: end-to-end coverage for both data-grounded and tool-using agents.

### Next Steps
- **Module 6** — Human review, labelling sessions, and judge calibration against human ground truth.
